# GPU acceleration with NVIDIA cuQuantum

scqubits provides optional support for NVIDIA cuQuantum to accelerate selected calculations for composite quantum systems on NVIDIA GPUs. Building on the [HilbertSpace](./hilbertspace.ipynb) workflow, this guide shows how to construct cuQuantum-backed operators and calculate low-energy eigenpairs. It also shows how scqubits-generated models can be used with qutip-cuquantum for GPU-accelerated dynamics.

## Preliminaries and dependencies

Using these features requires an NVIDIA GPU with a compatible CUDA driver and a working installation of [qutip-cuquantum](https://github.com/qutip/qutip-cuquantum#installation). qutip-cuquantum relies on NVIDIA cuQuantum Python (including cuDensityMat) and CuPy at runtime; these optional packages are not installed with scqubits. Follow the qutip-cuquantum installation instructions for the CUDA version available on your system.

```{=rst}
.. warning::

   Run all scqubits code outside ``CuQuantumBackend``. scqubits already manages
   the shared workstream internally for its cuQuantum operations. Activating
   the backend changes QuTiP defaults that scqubits does not support. Use the
   backend only around qutip-cuquantum operations; scqubits calls may be made
   before entering or after exiting the backend context.

```

We can verify the installation and confirm that a CUDA-capable GPU is available as follows:

In [ ]:
import cupy
import cuquantum
import qutip_cuquantum as qcu

device_id = cupy.cuda.Device().id
device_name = cupy.cuda.runtime.getDeviceProperties(device_id)["name"].decode()
print(f"CUDA device {device_id}: {device_name}")

CUDA device 0: NVIDIA GeForce RTX 5090


## cuQuantum features in scqubits

scqubits provides cuQuantum support for constructing composite operators and calculating partial eigensystems. These features are enabled explicitly and do not change the default CPU behavior.

### Composite Hamiltonians

`HilbertSpace.hamiltonian` can construct a composite Hamiltonian in the qutip-cuquantum representation used by cuDensityMat. The result remains a QuTiP `Qobj`, with its data represented by a `CuOperator`.

```python
hamiltonian = hilbert_space.hamiltonian(use_cuquantum=True)
```

### Identity-wrapped operators

`identity_wrap` expresses an operator acting on a subsystem in the bare product basis of the composite Hilbert space. Setting `use_cuquantum=True` constructs the resulting composite operator in the same qutip-cuquantum representation.

```python
wrapped_operator = scq.identity_wrap(
    subsystem_operator,
    subsystem,
    hilbert_space.subsystem_list,
    use_cuquantum=True,
)
```

### Eigensystems

scqubits supports sparse diagonalization of composite `HilbertSpace` systems through cuDensityMat's Krylov eigensolver, which computes a subset of the spectrum. The cuQuantum eigensolver is not supported for individual scqubits subsystems such as qubits. Set `esys_method` to `"esys_cuquantum"` to calculate a requested number of low-energy eigenpairs.

```python
hilbert_space.esys_method = "esys_cuquantum"
evals, evecs = hilbert_space.eigensys(evals_count=5)
```

When only eigenvalues are needed, use `evals_method="evals_cuquantum"` with `HilbertSpace.eigenvals`. See [Eigensolver settings and limitations](#Eigensolver-settings-and-limitations) for the Krylov parameters and restrictions on the number of requested eigenpairs.

### Workstream management

cuDensityMat objects must use the same `WorkStream` to interact with one another. scqubits therefore uses a single workstream for its cuQuantum operations. `get_cuquantum_workstream` creates the default workstream on first use and returns the same instance on subsequent calls. To create compatible objects outside scqubits, retrieve and reuse this workstream:

```python
workstream = scq.get_cuquantum_workstream()
```

If objects created outside scqubits already use a workstream, register that same instance with scqubits before using its cuQuantum features:

```python
scq.set_cuquantum_workstream(custom_workstream)
```

Registration must occur before the first call to `get_cuquantum_workstream` or any scqubits cuQuantum operation. Once initialized, the workstream cannot be replaced during the same Python process.

## `HilbertSpace` workflow with cuQuantum

The usual scqubits workflow for defining a composite system is unchanged when using cuQuantum. As an example, consider a fluxonium charge coupled to a harmonic resonator,

\begin{equation}
H = H_\mathrm{f} + H_\mathrm{r} + g n_\mathrm{f} n_\mathrm{r}.
\end{equation}

We first create the subsystems and add the interaction to a `HilbertSpace`:

In [ ]:
fluxonium = scq.Fluxonium(
    EJ=3.395,
    EC=1.0,
    EL=0.132,
    flux=0.5,
    cutoff=110,
    truncated_dim=20,
)
resonator = scq.Oscillator(
    E_osc=5.5,
    l_osc=1.0,
    truncated_dim=50,
)

hilbert_space = scq.HilbertSpace([fluxonium, resonator])
hilbert_space.add_interaction(
    g_strength=0.1,
    op1=fluxonium.n_operator,
    op2=resonator.n_operator,
)

### Hamiltonian and composite operators

Passing `use_cuquantum=True` constructs the Hamiltonian with the qutip-cuquantum data layer, using scqubits' shared workstream. If no workstream has been registered, a default workstream is created automatically.

In [ ]:
hamiltonian = hilbert_space.hamiltonian(use_cuquantum=True)
print(type(hamiltonian))
print(type(hamiltonian.data))

<class 'qutip.core.qobj.Qobj'>
<class 'qutip_cuquantum.operator.CuOperator'>


We use `identity_wrap` to express the resonator charge operator in the composite Hilbert space for the drive used below.

In [ ]:
resonator_charge = scq.identity_wrap(
    resonator.n_operator(),
    resonator,
    hilbert_space.subsystem_list,
    use_cuquantum=True,
)
type(resonator_charge.data)

qutip_cuquantum.operator.CuOperator

### Low-energy eigensystem

We use `generate_lookup` to calculate ten low-energy eigenpairs and assign bare-state labels using bare-energy ordering. With `esys_method="esys_cuquantum"`, it internally calls `eigensys` using the cuQuantum eigensolver.

In [ ]:
hilbert_space.esys_method = "esys_cuquantum"
hilbert_space.generate_lookup(ordering="BE", BEs_count=10)
evals = hilbert_space["evals"][0]
evecs = hilbert_space["evecs"][0]
evals

array([-0.46386953, -0.33372671,  3.17180909,  4.09615664,  4.47101911,
        4.62471681,  5.03748573,  5.16673276,  6.43217623,  7.65245281])

### Resonator charge-drive dynamics

The cuQuantum-backed Hamiltonian and operators can be passed to qutip-cuquantum for time evolution. We drive the resonator at the dressed $|0,0\rangle \rightarrow |0,1\rangle$ transition, starting from the dressed ground state. We use the eigenpairs calculated above to set the drive frequency and initial state:

In [ ]:
drive_frequency = hilbert_space.energy_by_bare_index(
    (0, 1), subtract_ground=True
)
ground_index = hilbert_space.dressed_index((0, 0))
initial_state = evecs[ground_index]
drive_frequency

np.float64(5.50135526151138)

The driven Hamiltonian is

\begin{equation}
H(t) = H + A \cos(2\pi f_\mathrm{d} t)n_\mathrm{r},
\end{equation}

We use a drive amplitude of $A=0.005$ GHz and evolve the closed system for 50 ns.

In [ ]:
drive_amplitude = 0.005
times = np.linspace(0.0, 50.0, 201)

def drive(t, amplitude, frequency):
    return amplitude * np.cos(2 * np.pi * frequency * t)

driven_hamiltonian = qt.QobjEvo(
    [
        2 * np.pi * hamiltonian,
        [2 * np.pi * resonator_charge, drive],
    ],
    args={
        "amplitude": drive_amplitude,
        "frequency": drive_frequency,
    },
)

Finally, we construct the resonator number operator to monitor its occupation and configure QuTiP's `SESolver` with the `CuVern7` integrator. We then call `solver.run` inside `CuQuantumBackend`, using scqubits' shared workstream.

In [ ]:
resonator_number = scq.identity_wrap(
    resonator.creation_operator() @ resonator.annihilation_operator(),
    resonator,
    hilbert_space.subsystem_list,
    use_cuquantum=True,
)
solver = qt.SESolver(
    driven_hamiltonian,
    options={
        "method": "CuVern7",
        "atol": 1e-8,
        "rtol": 1e-8,
        "nsteps": 10_000,
    },
)
workstream = scq.get_cuquantum_workstream()

with qcu.CuQuantumBackend(workstream):
    result = solver.run(
        initial_state,
        tlist=times,
        e_ops=[resonator_number],
    )

occupation = np.asarray(result.expect[0])
print(f"Initial resonator occupation: {occupation[0]:.9g}")
print(f"Maximum resonator occupation: {occupation.max():.9g}")
print(f"Final resonator occupation: {occupation[-1]:.9g}")

Initial resonator occupation: 1.04974128e-05
Maximum resonator occupation: 0.307033729
Final resonator occupation: 0.307033729


## Eigensolver settings and limitations

### Krylov settings

The cuQuantum eigensolver reads the following attributes from `scq.settings` on each call. The settings apply to both `evals_cuquantum` and `esys_cuquantum`.

| Setting | scqubits default | Meaning |
| --- | --- | --- |
| `CUQUANTUM_MIN_KRYLOV_BLOCK_SIZE` ($b$) | `1` | Minimum number of vectors in a Krylov block. |
| `CUQUANTUM_MAX_BUFFER_RATIO` ($r$) | `5` | Maximum ratio of Krylov subspace blocks to requested eigenpairs. |
| `CUQUANTUM_MAX_RESTARTS` | `20` | Maximum number of restart cycles. |

Larger blocks and subspaces may improve convergence at the cost of additional memory and computation. Increasing the restart limit allows more iterations. The buffer ratio must be greater than one. See NVIDIA's [OperatorSpectrumConfig documentation](https://docs.nvidia.com/cuda/cuquantum/latest/python/generated/cuquantum.densitymat.OperatorSpectrumConfig.html) for details.

Set these attributes before calling `eigensys`, `eigenvals`, or `generate_lookup`.

### Partial spectra and lookup generation

The Krylov solver is intended for a subset of low-energy eigenpairs. For a composite Hilbert space of dimension $D$, the maximum number of eigenpairs a user can request is

\begin{equation}
k_\mathrm{max} = \left\lfloor \frac{D-b}{2br} \right\rfloor,
\end{equation}

Increasing the minimum Krylov block size ($b$) or maximum buffer ratio ($r$) can lower this limit. Set `evals_count` to no more than $k_\mathrm{max}$ when calling `eigenvals` or `eigensys`.

For `generate_lookup`, use `ordering="BE"` and explicitly set `BEs_count` to the number of eigenpairs to calculate, also no greater than $k_\mathrm{max}$. Call `generate_lookup` outside `CuQuantumBackend`, as shown in the workflow above.

The `"DE"` and `"LX"` lookup orderings require a full eigensystem and are not supported with `esys_cuquantum`. Omitting `BEs_count` also requests a full eigensystem. If a full spectrum or either of these orderings is needed, select an alternative method described in [User-defined Diagonalization](../../settings/ipynb/custom_diagonalization.ipynb).